🚧 Advanced Build 🚧

Create an Agent that can help make PRDs (Product Requirement Documents) that you can use in Cursor as "project rules" (`.cursor/rules/new-prd.mdc`) for the development of your Demo Day Project.

In [2]:
import os 
import getpass
import asyncio
import nest_asyncio
import time
from pydantic import BaseModel
from agents import Agent
from agents import WebSearchTool
from agents.model_settings import ModelSettings
from typing import Any
from rich.console import Console, Group
from rich.live import Live
from rich.spinner import Spinner
from __future__ import annotations

from agents import Runner, custom_span, gen_trace_id, trace

nest_asyncio.apply()
os.environ["OPENAI_API_KEY"] = getpass.getpass()

In [3]:
class WebSearchItem(BaseModel):
    reason: str
    "Your reasoning for why this search is important to the query."

    query: str
    "The search term to use for the web search."

class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem]
    """A list of web searches to perform to best answer the query."""


class ReportData(BaseModel):
    short_summary: str
    """A short 2-3 sentence summary of the findings."""

    markdown_report: str
    """The final report"""

    follow_up_questions: list[str]
    """Suggested topics to research further"""

class Printer:
    def __init__(self, console: Console):
        self.live = Live(console=console)
        self.items: dict[str, tuple[str, bool]] = {}
        self.hide_done_ids: set[str] = set()
        self.live.start()

    def end(self) -> None:
        self.live.stop()

    def hide_done_checkmark(self, item_id: str) -> None:
        self.hide_done_ids.add(item_id)

    def update_item(
        self, item_id: str, content: str, is_done: bool = False, hide_checkmark: bool = False
    ) -> None:
        self.items[item_id] = (content, is_done)
        if hide_checkmark:
            self.hide_done_ids.add(item_id)
        self.flush()

    def mark_item_done(self, item_id: str) -> None:
        self.items[item_id] = (self.items[item_id][0], True)
        self.flush()

    def flush(self) -> None:
        renderables: list[Any] = []
        for item_id, (content, is_done) in self.items.items():
            if is_done:
                prefix = "✅ " if item_id not in self.hide_done_ids else ""
                renderables.append(prefix + content)
            else:
                renderables.append(Spinner("dots", text=content))
        self.live.update(Group(*renderables))

# prompts

PLANNER_PROMPT = (
    "You are a product manager and market researcher. Given a product idea or feature request, come up with a set of web searches"
    "to understand market needs, competitive landscape, user requirements, and technical feasibility. "
    "Focus on searches that will help create a comprehensive Product Requirements Document (PRD). "
    "Output between 5 and 20 search terms covering market research, user needs, competitors, and technical considerations."
)

SEARCH_PROMPT = (
    "You are a product research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results relevant to product development and market analysis. "
    "The summary must be 2-3 paragraphs and less than 300 words. Capture the main points about market trends, "
    "user needs, competitive insights, or technical considerations. Write succinctly, no need to have complete "
    "sentences or good grammar. This will be consumed by someone creating a PRD, so it's vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary itself."
)

WRITER_PROMPT = (
    "You are a senior product manager tasked with creating a comprehensive Product Requirements Document (PRD) "
    "and corresponding .cursor rules for development. You will be provided with the original product query, "
    "and research findings from market analysis.\n\n"
    "Your task is to:\n"
    "1. Create a detailed PRD in markdown format that includes:\n"
    "   - Executive Summary\n"
    "   - Product Overview and Vision\n"
    "   - User Personas and Use Cases\n"
    "   - Functional Requirements\n"
    "   - Non-Functional Requirements\n"
    "   - Technical Architecture Overview\n"
    "   - Success Metrics and KPIs\n"
    "   - Timeline and Milestones\n"
    "   - Risk Assessment\n\n"
    "2. Generate .cursor rules (in a separate section) that will help developers create a prototype. "
    "These rules should include:\n"
    "   - Technology stack recommendations\n"
    "   - Code structure and architecture guidelines\n"
    "   - Development best practices\n"
    "   - Key features to implement first\n"
    "   - Testing and deployment considerations\n\n"
    "The PRD should be comprehensive (5-10 pages, at least 1000 words) and the .cursor rules should be "
    "actionable for immediate development kickoff. Format the .cursor rules as a separate markdown section "
    "that can be directly copied to .cursor/rules/new-prd.mdc"
)

# agents

planner_agent = Agent(
    name="PlannerAgent",
    instructions=PLANNER_PROMPT,
    model="gpt-4.1",
    output_type=WebSearchPlan,
)

search_agent = Agent(
    name="Search agent",
    instructions=SEARCH_PROMPT,
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required"),
)

writer_agent = Agent(
    name="WriterAgent",
    instructions=WRITER_PROMPT,
    model="o3-mini",
    output_type=ReportData,
)

In [4]:
class ResearchManager:
    def __init__(self):
        self.console = Console()
        self.printer = Printer(self.console)

    async def run(self, query: str) -> None:
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            self.printer.update_item(
                "trace_id",
                f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}",
                is_done=True,
                hide_checkmark=True,
            )

            self.printer.update_item(
                "starting",
                "Starting research...",
                is_done=True,
                hide_checkmark=True,
            )
            search_plan = await self._plan_searches(query)
            search_results = await self._perform_searches(search_plan)
            report = await self._write_report(query, search_results)

            final_report = f"Report summary\n\n{report.short_summary}"
            self.printer.update_item("final_report", final_report, is_done=True)

            self.printer.end()

        print("\n\n=====REPORT=====\n\n")
        print(f"Report: {report.markdown_report}")
        print("\n\n=====FOLLOW UP QUESTIONS=====\n\n")
        unique_questions = []
        seen = set()
        
        for question in report.follow_up_questions:
            if question not in seen:
                unique_questions.append(question)
                seen.add(question)
        
        for i, question in enumerate(unique_questions, 1):
            print(f"{i}. {question}")

    async def _plan_searches(self, query: str) -> WebSearchPlan:
        self.printer.update_item("planning", "Planning searches...")
        result = await Runner.run(
            planner_agent,
            f"Query: {query}",
        )
        self.printer.update_item(
            "planning",
            f"Will perform {len(result.final_output.searches)} searches",
            is_done=True,
        )
        return result.final_output_as(WebSearchPlan)

    async def _perform_searches(self, search_plan: WebSearchPlan) -> list[str]:
        with custom_span("Search the web"):
            self.printer.update_item("searching", "Searching...")
            num_completed = 0
            max_concurrent = 5
            results = []
            
            for i in range(0, len(search_plan.searches), max_concurrent):
                batch = search_plan.searches[i:i+max_concurrent]
                tasks = [asyncio.create_task(self._search(item)) for item in batch]
                
                for task in asyncio.as_completed(tasks):
                    try:
                        result = await task
                        if result is not None:
                            results.append(result)
                    except Exception as e:
                        print(f"Search error: {e}")
                        
                    num_completed += 1
                    self.printer.update_item(
                        "searching", f"Searching... {num_completed}/{len(search_plan.searches)} completed"
                    )
            
            self.printer.mark_item_done("searching")
            return results

    async def _search(self, item: WebSearchItem) -> str | None:
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        try:
            result = await Runner.run(
                search_agent,
                input,
            )
            return str(result.final_output)
        except Exception as e:
            print(f"Error searching for '{item.query}': {e}")
            return None

    async def _write_report(self, query: str, search_results: list[str]) -> ReportData:
        self.printer.update_item("writing", "Thinking about report...")
        input = f"Original query: {query}\nSummarized search results: {search_results}"
        
        result = Runner.run_streamed(
            writer_agent,
            input,
        )
        
        update_messages = [
            "Thinking about report...",
            "Planning report structure...",
            "Writing outline...",
            "Creating sections...",
            "Cleaning up formatting...",
            "Finalizing report...",
            "Finishing report...",
        ]

        last_update = time.time()
        next_message = 0
        
        async for event in result.stream_events():
            if time.time() - last_update > 5 and next_message < len(update_messages):
                self.printer.update_item("writing", update_messages[next_message])
                next_message += 1
                last_update = time.time()

        self.printer.mark_item_done("writing")
        return result.final_output_as(ReportData)

In [5]:
async def main() -> None:
    query = input("A web application that allows users to create and share their own AI agents")
    await ResearchManager().run(query)

asyncio.run(main())

Output()



=====REPORT=====


Report: # Product Requirements Document (PRD)

## 1. Executive Summary

The goal of this product is to capture a realistic segment of the market by addressing real user pain points and unmet needs. Our approach is grounded in thorough market research and competitive analysis. By assessing Total Addressable Market (TAM), Serviceable Available Market (SAM), and Serviceable Obtainable Market (SOM), we ensure that this product is designed with scalability in mind, appealing to both early adopters and mainstream users. The product will integrate modern technologies including AI/ML, IoT, and AR/VR functionalities, complementing a flexible pricing strategy with models such as subscription, freemium, and bundle pricing. The development will adhere to regulatory standards and will use robust planning for integration, testing, and deployment.

## 2. Product Overview and Vision

### Vision Statement

Our vision is to deliver a cutting-edge product that not only satisfies a cl